# AI-Powered Skin Cancer Detection and Patient Triage Support System
**Objective:** Build a production-ready, explainable AI-powered healthcare system using HAM10000 dataset that classifies skin lesions, provides risk assessment, personalized recommendations, and hospital guidance.

In [ ]:
# CELL 1: Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import glob
import os
import pickle
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# CELL 2: Load metadata CSV
# Update path as per your Colab environment
metadata_path = 'HAM10000_metadata.csv'
df = pd.read_csv(metadata_path)

print("Metadata Shape:", df.shape)
display(df.head())
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# CELL 3: Combine image folders and create image path dictionary
image_paths = {}
for x in glob.glob(os.path.join('HAM10000_images_part_1', '*.jpg')):
    image_paths[os.path.splitext(os.path.basename(x))[0]] = x

for x in glob.glob(os.path.join('HAM10000_images_part_2', '*.jpg')):
    image_paths[os.path.splitext(os.path.basename(x))[0]] = x

print(f"Total images found: {len(image_paths)}")

In [ ]:
# CELL 4: Create columns and Display class distribution
df['path'] = df['image_id'].map(image_paths.get)
df['cell_type'] = df['dx'] # dx is the class column in HAM10000

plt.figure(figsize=(10,6))
sns.countplot(x='cell_type', data=df, palette='viridis')
plt.title('Class Distribution')
plt.show()

print("Class Counts:\n", df['cell_type'].value_counts())

In [ ]:
# CELL 5: Label encoding
encoder = LabelEncoder()
df['encoded_label'] = encoder.fit_transform(df['cell_type'])

print("Encoded Classes:")
for i, label in enumerate(encoder.classes_):
    print(f"{i}: {label}")

In [ ]:
# CELL 6: Train-validation-test split (70-15-15)
# Split only dataframe to save memory
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['cell_type'])
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42, stratify=test_df['cell_type'])

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

In [ ]:
# CELL 7: Create ImageDataGenerators
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='path',
    y_col='cell_type',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='path',
    y_col='cell_type',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='path',
    y_col='cell_type',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
# CELL 8: MobileNetV2 Model, Training, Evaluation, and Triage System

# 1. Risk Assessment Configuration
CLASS_INFO = {
    'nv': {'risk': 'GREEN', 'name': 'Melanocytic Nevi', 'description': 'Common benign moles'}, 
    'bkl': {'risk': 'YELLOW', 'name': 'Benign Keratosis', 'description': 'Non-cancerous growths'}, 
    'vasc': {'risk': 'YELLOW', 'name': 'Vascular Lesions', 'description': 'Blood vessel-related lesions'}, 
    'df': {'risk': 'YELLOW', 'name': 'Dermatofibroma', 'description': 'Benign skin nodules'}, 
    'akiec': {'risk': 'RED', 'name': 'Actinic Keratoses', 'description': 'Precancerous lesions'}, 
    'bcc': {'risk': 'RED', 'name': 'Basal Cell Carcinoma', 'description': 'Common skin cancer'}, 
    'mel': {'risk': 'RED', 'name': 'Melanoma', 'description': 'Dangerous skin cancer'}
}

# 2. Compute class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['encoded_label']),
    y=train_df['encoded_label']
)
class_weights_dict = dict(enumerate(class_weights))

# 3. Build MobileNetV2 Model
input_layer = Input(shape=(224, 224, 3))
base_mobilenet = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_mobilenet.trainable = False

x = base_mobilenet(input_layer)
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
output_layer = Dense(7, activation='softmax')(x)

model = Model(inputs=input_layer, outputs=output_layer)

# 4. Compile and Callbacks
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.2, patience=3, monitor='val_loss'),
    ModelCheckpoint('skin_cancer_best.keras', save_best_only=True, monitor='val_loss')
]

# 5. Training Stage 1: Frozen MobileNetV2
print("Starting Training Stage 1 (Frozen Backbone)...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=40,
    class_weight=class_weights_dict,
    callbacks=callbacks
)

# 6. Training Stage 2: Fine-tuning (Unfreeze top 50 layers)
print("\nStarting Training Stage 2 (Fine-tuning)...")
base_mobilenet.trainable = True
for layer in base_mobilenet.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    class_weight=class_weights_dict,
    callbacks=callbacks
)

# 7. Evaluation
def plot_history(history, history_finetune):
    acc = history.history['accuracy'] + history_finetune.history['accuracy']
    val_acc = history.history['val_accuracy'] + history_finetune.history['val_accuracy']
    loss = history.history['loss'] + history_finetune.history['loss']
    val_loss = history.history['val_loss'] + history_finetune.history['val_loss']
    
    epochs_range = range(len(acc))
    
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.title('Accuracy Graphs')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.title('Loss Graphs')
    plt.legend()
    plt.show()

plot_history(history, history_finetune)

# Test Accuracy & Report
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc*100:.2f}%")

Y_pred = model.predict(test_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = test_generator.classes

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=encoder.classes_))

plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues', 
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# 8. Save Artifacts
model.save('skin_cancer_model.h5')
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)

# 9. Healthcare Support Functions
def get_gradcam(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap)
    return heatmap.numpy()

def get_personalized_recommendations(pred_class, confidence):
    recs = []
    info = CLASS_INFO[pred_class]
    
    if info['risk'] == 'RED':
        recs.append("🔴 URGENT: Schedule an appointment with a dermatologist within 1-2 weeks.")
        recs.append("🔴 Avoid sun exposure and use SPF 50+ sunscreen.")
        recs.append("🔴 Do not scratch or irritate the lesion.")
    elif info['risk'] == 'YELLOW':
        recs.append("🟡 MODERATE: Visit a dermatologist within 1-2 months for a checkup.")
        recs.append("🟡 Monitor the lesion for any changes in size, color, or shape.")
    else:
        recs.append("🟢 LOW RISK: The lesion appears benign.")
        recs.append("🟢 Continue regular self-examinations.")
    
    recs.append("📊 Daily sunscreen use (SPF 30+).")
    recs.append("🕶️ Wear protective clothing and avoid peak sun hours (10 AM - 4 PM).")
    return recs

def get_hospital_guidance():
    return [
        "🏥 AIIMS - All India Institute of Medical Sciences, New Delhi",
        "🏥 Apollo Hospitals - Nationwide",
        "🏥 Fortis Healthcare - Nationwide",
        "🏥 Tata Memorial Hospital, Mumbai",
        "🏥 Regional Cancer Centers (RCCs) across India"
    ]

def generate_report(img_path, model, encoder):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (224, 224))
    img_array = np.expand_dims(img_resized, axis=0)
    img_array = preprocess_input(img_array)
    
    preds = model.predict(img_array)
    pred_idx = np.argmax(preds[0])
    pred_class = encoder.classes_[pred_idx]
    confidence = preds[0][pred_idx]
    class_info = CLASS_INFO[pred_class]
    
    # Print Report
    print("#" * 60)
    print("#          SKIN CANCER DETECTION REPORT")
    print("#" * 60)
    print(f"Predicted Class: {class_info['name']} ({pred_class})")
    print(f"Confidence Score: {confidence*100:.2f}%")
    print(f"Risk Level: {class_info['risk']}")
    print(f"Description: {class_info['description']}")
    print("\n--- PERSONALIZED RECOMMENDATIONS ---")
    for rec in get_personalized_recommendations(pred_class, confidence):
        print(f"- {rec}")
    print("\n--- HOSPITAL GUIDANCE ---")
    for hosp in get_hospital_guidance():
        print(f"- {hosp}")
    
    # Visualization
    try:
        # Find last conv layer in MobileNetV2
        last_conv_layer = None
        for layer in reversed(base_mobilenet.layers):
            if isinstance(layer, tf.keras.layers.Conv2D):
                last_conv_layer = layer.name
                break
        if last_conv_layer is None:
            raise ValueError("Could not find Conv2D layer in model!")
            
        heatmap = get_gradcam(img_array, model, last_conv_layer)
        
        heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
        heatmap = np.uint8(255 * heatmap)
        heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
        superimposed_img = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)
        
        plt.figure(figsize=(15, 6))
        plt.subplot(1, 3, 1)
        plt.imshow(img)
        plt.title('Original Image')
        plt.subplot(1, 3, 2)
        plt.imshow(heatmap)
        plt.title('Grad-CAM Heatmap')
        plt.subplot(1, 3, 3)
        plt.imshow(superimposed_img)
        plt.title(f'Overlay: {pred_class}')
        plt.show()
    except Exception as e:
        print(f"\nNote: Could not generate Grad-CAM visualization: {e}")
        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.title(f'Prediction: {pred_class}')
        plt.show()
    print("#" * 60)
    return pred_class, confidence

# Example usage (uncomment and replace with actual path):
# generate_report(test_df.iloc[0]['path'], model, encoder)